In [1]:
# import numpy as np
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()


# Dense Layer
class Layer_Dense:
    def __init__(self, input_count, neurons_count):
        self.weights = 0.01 * np.random.randn(input_count, neurons_count)
        self.biases = np.zeros((1, neurons_count))

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        self.dinputs = np.dot(dvalues, self.weights.T)

    def __repr__(self):
        weights_str = np.array2string(self.weights, precision=6, suppress_small=True)
        biases_str = np.array2string(self.biases, precision=6, suppress_small=True)
        return f"Layer_Dense(\n  weights={weights_str},\n  biases={biases_str}\n)"


class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

    def __repr__(self):
        parts = []
        if hasattr(self, "inputs") and isinstance(self.inputs, np.ndarray):
            parts.append(f"inputs.shape={self.inputs.shape}")
        if hasattr(self, "output") and isinstance(self.output, np.ndarray):
            parts.append(f"output.shape={self.output.shape}")
        if hasattr(self, "dinputs") and isinstance(self.dinputs, np.ndarray):
            parts.append(f"dinputs.shape={self.dinputs.shape}")
        inner = ", ".join(parts)
        return (
            f"{self.__class__.__name__}({inner})"
            if inner
            else f"{self.__class__.__name__}()"
        )


class Activation_Softmax:
    def forward(self, inputs):
        self.inputs = inputs
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output = probabilities

    def backward(self, dvalues):
        self.dinputs = np.empty_like(dvalues)

        for index, (single_output, single_dvalues) in enumerate(zip(self.output, dvalues)):
            single_output = single_output.reshape(-1, 1)
            jacobian_matrix = np.diaflat(single_output) - np.dot(single_output, single_output.T)
            self.dinputs[index] = np.dot(jacobian_matrix, single_dvalues)

    def __repr__(self):
        parts = []
        if hasattr(self, "inputs") and isinstance(self.inputs, np.ndarray):
            parts.append(f"inputs.shape={self.inputs.shape}")
        if hasattr(self, "output") and isinstance(self.output, np.ndarray):
            parts.append(f"output.shape={self.output.shape}")
        if hasattr(self, "dinputs") and isinstance(self.dinputs, np.ndarray):
            parts.append(f"dinputs.shape={self.dinputs.shape}")
        inner = ", ".join(parts)
        return (
            f"{self.__class__.__name__}({inner})"
            if inner
            else f"{self.__class__.__name__}()"
        )


class Loss:
    def calculate(self, output, y):
        sample_losses = self.forward(output, y)
        data_loss = np.mean(sample_losses)
        return data_loss

    def __repr__(self):
        return f"{self.__class__.__name__}()"


class Loss_CategoricalCrossEntropy(Loss):
    def forward(self, y_pred, y_true):
        samples = len(y_pred)
        y_pred_clipped = np.clip(y_pred, 1e-7, 1-1e-7)
        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)

        negative_log_likelihoods = -np.log(correct_confidences)
        return negative_log_likelihoods

    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        labels = len(dvalues[0])

        if len(y_true.shape) == 1:
            y_true = np.eye(labels)[y_true]

        self.dinputs = -y_true / dvalues
        self.dinputs = self.dinputs / samples

    def __repr__(self):
        parts = []
        if hasattr(self, "dinputs") and isinstance(self.dinputs, np.ndarray):
            parts.append(f"dinputs.shape={self.dinputs.shape}")
        inner = ", ".join(parts)
        return (
            f"{self.__class__.__name__}({inner})"
            if inner
            else f"{self.__class__.__name__}()"
        )


class Activation_Softmax_Loss_CategroicalCrossEntropy():
    def __init__(self):
        self.activation = Activation_Softmax()
        self.loss = Loss_CategoricalCrossEntropy()

    def forward(self, inputs, y_true):
        self.activation.forward(inputs)
        self.output = self.activation.output
        return self.loss.calculate(self.output, y_true)

    def backward(self, dvalues, y_true):
        samples = len(dvalues)
        if len(y_true.shape) == 2:
            y_true = np.argmax(y_true, axis=1)

        self.dinputs = dvalues.copy()
        self.dinputs[range(samples), y_true] -= 1
        self.dinputs = self.dinputs / samples

    def __repr__(self):
        parts = []
        if hasattr(self, "output") and isinstance(self.output, np.ndarray):
            parts.append(f"output.shape={self.output.shape}")
        if hasattr(self, "dinputs") and isinstance(self.dinputs, np.ndarray):
            parts.append(f"dinputs.shape={self.dinputs.shape}")
        parts.append(f"activation={self.activation.__class__.__name__}")
        parts.append(f"loss={self.loss.__class__.__name__}")
        inner = ", ".join(parts)
        return f"{self.__class__.__name__}({inner})"


X, y = spiral_data(samples=100, classes=3)
X, y

(array([[ 0.00000000e+00,  0.00000000e+00],
        [ 2.99555692e-03,  9.64660756e-03],
        [ 1.28809698e-02,  1.55628482e-02],
        [ 2.99747903e-02,  4.44809627e-03],
        [ 3.93124595e-02,  9.32828337e-03],
        [ 8.28829070e-04,  5.04982509e-02],
        [ 5.34835160e-02,  2.85062827e-02],
        [ 4.17361967e-02,  5.70752136e-02],
        [ 5.54633923e-02,  5.87686822e-02],
        [ 8.16038325e-02,  4.00659069e-02],
        [ 8.91875103e-02,  4.74197045e-02],
        [ 1.07160836e-01, -2.93638241e-02],
        [ 1.21183202e-01, -2.64751446e-03],
        [ 1.28777727e-01,  2.56794747e-02],
        [ 1.41112968e-01, -9.22449492e-03],
        [ 1.50579467e-01, -1.68126337e-02],
        [ 1.13476366e-01, -1.15077794e-01],
        [ 1.71552509e-01, -7.51816481e-03],
        [ 1.67186841e-01, -7.14591444e-02],
        [ 1.91325873e-01,  1.50793232e-02],
        [ 1.36771902e-01,  1.48679554e-01],
        [ 1.35606423e-01, -1.63114399e-01],
        [ 1.04024746e-01, -1.963

In [2]:
dense1 = Layer_Dense(2, 3)
activation1 = Activation_ReLU()

dense2 = Layer_Dense(3, 3)
loss_activation = Activation_Softmax_Loss_CategroicalCrossEntropy()

In [3]:
dense1.forward(X)
dense1

Layer_Dense(
  weights=[[-0.013065  0.016581 -0.001182]
 [-0.006802  0.006664 -0.004607]],
  biases=[[0. 0. 0.]]
)

In [4]:
activation1.forward(dense1.output)
activation1

Activation_ReLU(inputs.shape=(300, 3), output.shape=(300, 3))

In [5]:
dense2.forward(activation1.output)
dense2

Layer_Dense(
  weights=[[-0.013343 -0.013467  0.006938]
 [-0.001596 -0.001337  0.010777]
 [-0.011268 -0.007307 -0.003849]],
  biases=[[0. 0. 0.]]
)

In [6]:
loss = loss_activation.forward(dense2.output, y)

print(loss_activation.output[:5])

print("Loss: ", loss)

[[0.33333334 0.33333334 0.33333334]
 [0.33333316 0.3333332  0.33333364]
 [0.33333287 0.3333329  0.33333418]
 [0.3333326  0.33333263 0.33333477]
 [0.33333233 0.3333324  0.33333528]]
Loss:  1.0986104


In [7]:
predictions = np.argmax(loss_activation.output, axis=1)
if len(y.shape) == 2:
    y = np.argmax(y, axis=1)
accuracy = np.mean(predictions == y)

print('accuracy: ', accuracy)

accuracy:  0.34


In [8]:
# Backward Pass
loss_activation.backward(loss_activation.output, y)
loss_activation

Activation_Softmax_Loss_CategroicalCrossEntropy(output.shape=(300, 3), dinputs.shape=(300, 3), activation=Activation_Softmax, loss=Loss_CategoricalCrossEntropy)

In [9]:
dense2.backward(loss_activation.dinputs)
dense2

Layer_Dense(
  weights=[[-0.013343 -0.013467  0.006938]
 [-0.001596 -0.001337  0.010777]
 [-0.011268 -0.007307 -0.003849]],
  biases=[[0. 0. 0.]]
)

In [10]:
activation1.backward(dense2.dinputs)
activation1

Activation_ReLU(inputs.shape=(300, 3), output.shape=(300, 3), dinputs.shape=(300, 3))

In [11]:
dense1.backward(activation1.dinputs)

# Print Gradients
print(dense1.dweights)
print(dense1.dbiases)
print(dense2.dweights)
print(dense2.dbiases)

[[ 1.5766358e-04  7.8368575e-05  4.7324407e-05]
 [ 1.8161036e-04  1.1045571e-05 -3.3096312e-05]]
[[-3.6055347e-04  9.6611722e-05 -1.0367142e-04]]
[[ 5.4410957e-05  1.0741142e-04 -1.6182236e-04]
 [-4.0791339e-05 -7.1678100e-05  1.1246944e-04]
 [-5.3011296e-05  8.5817286e-05 -3.2805994e-05]]
[[-1.0732794e-05 -9.4590941e-06  2.0027626e-05]]
